# 🔬 Data Science — Ejercicios con Soluciones

Este notebook cubre dos pilares fundamentales de la Ciencia de Datos:
- **Ejercicio 1:** Análisis de ventas con NumPy y pandas (el ciclo completo: cargar → limpiar → analizar → visualizar)
- **Ejercicio 2:** Optimización matemática con SciPy (encontrar el máximo/mínimo de funciones con restricciones)

**Librerías necesarias:**
```bash
pip install numpy pandas matplotlib seaborn scipy
```

**Cómo usar este notebook:**
1. Lee el **enunciado** y la **explicación conceptual**.
2. Intentá resolver en la celda `# TU CÓDIGO AQUÍ`.
3. Si necesitás ayuda, desplegá la **💡 Pista**.
4. Comparate con la **✅ Solución**.

---

In [ ]:
# 🔧 CELDA DE SETUP — Ejecutá esto primero
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import scipy
from scipy import optimize, stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Librerías cargadas:')
print(f'   numpy   {np.__version__}')
print(f'   pandas  {pd.__version__}')
print(f'   scipy   {scipy.__version__}')

---
## 📦 Ejercicio 1 — Análisis de ventas con NumPy y pandas

**Enunciado:**
- Cargá un dataset de ventas con pandas y NumPy.
- Calculá el total de ventas por producto.
- Encontrá el producto más vendido.
- Creá un gráfico de barras para visualizar las ventas por producto.

### 📖 Explicación conceptual

#### NumPy — El motor numérico
NumPy provee **arrays** (arreglos multidimensionales) que son mucho más rápidos que las listas de Python para operaciones matemáticas. La clave es que las operaciones son **vectorizadas**: se aplican a todos los elementos a la vez, sin bucles.

```python
import numpy as np

a = np.array([1, 2, 3, 4, 5])
a * 2          # → [2, 4, 6, 8, 10]  (sin for loop)
a[a > 3]       # → [4, 5]            (filtrado booleano)
np.sum(a)      # → 15
np.mean(a)     # → 3.0
np.std(a)      # → 1.41
```

#### pandas — El análisis de datos
pandas construye sobre NumPy y agrega estructura: etiquetas, índices, y operaciones de tipo SQL.

| Operación | Código pandas |
|-----------|---------------|
| Agrupar y sumar | `df.groupby('producto')['ventas'].sum()` |
| El mayor | `.idxmax()` → nombre | `.max()` → valor |
| Ordenar | `.sort_values(ascending=False)` |
| Nueva columna | `df['total'] = df['cantidad'] * df['precio']` |
| Percentil | `df['ventas'].quantile(0.75)` |
| Tabla pivot | `pd.pivot_table(df, values='total', index='mes', columns='producto')` |

In [ ]:
# 🔧 GENERAMOS EL DATASET DE VENTAS
np.random.seed(2024)

n = 500
productos  = ['Laptop', 'Monitor', 'Teclado', 'Mouse', 'Auriculares', 'Webcam', 'SSD', 'Hub USB']
regiones   = ['Norte', 'Sur', 'Centro', 'Este', 'Oeste']
vendedores = ['Ana', 'Carlos', 'María', 'Pedro', 'Lucía']

# Precios base por producto
precios_base = {
    'Laptop': 1200, 'Monitor': 350, 'Teclado': 110,
    'Mouse': 25, 'Auriculares': 85, 'Webcam': 60, 'SSD': 120, 'Hub USB': 18
}

producto_col  = np.random.choice(productos, n, p=[0.15,0.12,0.14,0.18,0.12,0.09,0.11,0.09])
cantidad_col  = np.random.randint(1, 15, n)
precio_col    = np.array([precios_base[p] * np.random.uniform(0.9, 1.1) for p in producto_col]).round(2)
fecha_col     = pd.date_range('2023-01-01', periods=n, freq='16H')

df_ventas = pd.DataFrame({
    'fecha':     fecha_col,
    'producto':  producto_col,
    'cantidad':  cantidad_col,
    'precio':    precio_col,
    'region':    np.random.choice(regiones, n),
    'vendedor':  np.random.choice(vendedores, n),
})
df_ventas['total'] = df_ventas['cantidad'] * df_ventas['precio']
df_ventas['mes']   = df_ventas['fecha'].dt.month
df_ventas['trim']  = df_ventas['fecha'].dt.quarter

df_ventas.to_csv('ventas_ds.csv', index=False)
print(f'Dataset generado: {df_ventas.shape[0]} registros, {df_ventas.shape[1]} columnas')
print(f'Período: {df_ventas["fecha"].min().date()} → {df_ventas["fecha"].max().date()}')
df_ventas.head()

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Cargá 'ventas_ds.csv' con pd.read_csv()
# 2. Calculá el total de ventas (suma de 'total') por producto
# 3. Encontrá el producto más vendido en unidades y en dinero
# 4. Usá NumPy para calcular estadísticas del array de ventas
# 5. Creá un gráfico de barras con las ventas por producto


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```python
df = pd.read_csv('ventas_ds.csv')

# Total de ventas por producto
ventas_prod = df.groupby('producto')['total'].sum().sort_values(ascending=False)

# El más vendido en dinero
mas_vendido = ventas_prod.idxmax()

# Estadísticas con NumPy
arr_totales = df['total'].values  # Convierte la columna a array NumPy
print(np.mean(arr_totales), np.std(arr_totales), np.percentile(arr_totales, 75))

# Gráfico
ventas_prod.plot(kind='bar')
```
**Bonus:** Usá `pd.pivot_table()` para ver las ventas por producto Y por mes en una sola tabla.

</details>

In [ ]:
# ✅ SOLUCIÓN — Parte A: Análisis con pandas y NumPy

df = pd.read_csv('ventas_ds.csv', parse_dates=['fecha'])

print('=== PASO 1: Resumen del dataset ===')
print(f'Registros: {len(df):,} | Columnas: {len(df.columns)}')
print(f'Nulos: {df.isnull().sum().sum()}')

# --- Análisis con pandas ---
print('\n=== PASO 2: Total de ventas por producto ===')
ventas_prod  = df.groupby('producto')['total'].sum().sort_values(ascending=False)
unidades_prod = df.groupby('producto')['cantidad'].sum().sort_values(ascending=False)

resumen_prod = pd.DataFrame({
    'ventas_$':    ventas_prod,
    'unidades':    unidades_prod,
    'ticket_prom': (ventas_prod / unidades_prod).round(2),
    'participacion_%': (ventas_prod / ventas_prod.sum() * 100).round(1)
}).sort_values('ventas_$', ascending=False)

print(resumen_prod.round(0).to_string())

print(f'\n=== PASO 3: Top productos ===')
print(f'💰 Más vendido en dinero:    {ventas_prod.idxmax()} (${ventas_prod.max():,.0f})')
print(f'📦 Más vendido en unidades:  {unidades_prod.idxmax()} ({unidades_prod.max():,} unidades)')
print(f'💎 Mayor ticket promedio:    {resumen_prod["ticket_prom"].idxmax()} (${resumen_prod["ticket_prom"].max():,.0f})')

# --- Estadísticas con NumPy ---
print('\n=== PASO 4: Estadísticas NumPy sobre el array de ventas ===')
arr = df['total'].values  # Convierte la Serie de pandas a ndarray de NumPy
print(f'Tipo:          {type(arr)}')
print(f'Shape:         {arr.shape}')
print(f'Media:         ${np.mean(arr):,.2f}')
print(f'Mediana:       ${np.median(arr):,.2f}')
print(f'Desv. estándar:${np.std(arr):,.2f}')
print(f'Percentil 25:  ${np.percentile(arr, 25):,.2f}')
print(f'Percentil 75:  ${np.percentile(arr, 75):,.2f}')
print(f'Mínimo:        ${np.min(arr):,.2f}')
print(f'Máximo:        ${np.max(arr):,.2f}')
print(f'Total general: ${np.sum(arr):,.2f}')

# Ventas por encima de la media (operación vectorizada)
sobre_media = arr[arr > np.mean(arr)]
print(f'\nVentas sobre la media: {len(sobre_media)} ({len(sobre_media)/len(arr)*100:.1f}%)')

In [ ]:
# ✅ SOLUCIÓN — Parte B: Dashboard de visualizaciones

fig = plt.figure(figsize=(17, 12))
colores_prod = sns.color_palette('muted', len(ventas_prod))

# 1. Ventas totales por producto (barras horizontales)
ax1 = fig.add_subplot(2, 3, 1)
ventas_ord = ventas_prod.sort_values()
barras = ax1.barh(ventas_ord.index, ventas_ord.values,
                  color=sns.color_palette('muted', len(ventas_ord)), edgecolor='white')
ax1.set_title('Ventas totales por producto ($)', fontweight='bold')
ax1.set_xlabel('Total de ventas ($)')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
for bar in barras:
    ax1.text(bar.get_width() + 500, bar.get_y() + bar.get_height()/2,
             f'${bar.get_width()/1000:.1f}K', va='center', fontsize=8)

# 2. Participación de mercado (torta)
ax2 = fig.add_subplot(2, 3, 2)
wedges, texts, autotexts = ax2.pie(
    ventas_prod.values, labels=ventas_prod.index,
    autopct='%1.1f%%', startangle=90,
    colors=colores_prod,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for t in autotexts:
    t.set_fontsize(8)
ax2.set_title('Participación en ventas (%)', fontweight='bold')

# 3. Evolución mensual de ventas
ax3 = fig.add_subplot(2, 3, 3)
ventas_mes = df.groupby('mes')['total'].sum()
ax3.plot(ventas_mes.index, ventas_mes.values, 'o-', color='steelblue', linewidth=2.5, markersize=7)
ax3.fill_between(ventas_mes.index, ventas_mes.values, alpha=0.15, color='steelblue')
ax3.axhline(ventas_mes.mean(), color='tomato', linestyle='--', alpha=0.7, label=f'Media: ${ventas_mes.mean()/1000:.1f}K')
ax3.set_title('Evolución mensual de ventas', fontweight='bold')
ax3.set_xlabel('Mes')
ax3.set_ylabel('Ventas ($)')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax3.set_xticks(range(1, 13))
ax3.legend()

# 4. Mapa de calor: ventas por producto y trimestre
ax4 = fig.add_subplot(2, 3, 4)
pivot = df.pivot_table(values='total', index='producto', columns='trim', aggfunc='sum').fillna(0)
pivot.columns = [f'Q{c}' for c in pivot.columns]
sns.heatmap(pivot/1000, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax4,
            cbar_kws={'label': 'Ventas ($K)'}, linewidths=0.5)
ax4.set_title('Ventas por producto y trimestre ($K)', fontweight='bold')
ax4.set_ylabel('')

# 5. Top 3 vendedores por producto
ax5 = fig.add_subplot(2, 3, 5)
ventas_vend = df.groupby('vendedor')['total'].sum().sort_values(ascending=False)
colors_vend = ['gold', 'silver', '#cd7f32', 'steelblue', 'steelblue']
ax5.bar(ventas_vend.index, ventas_vend.values, color=colors_vend, edgecolor='white')
ax5.set_title('Ranking de vendedores', fontweight='bold')
ax5.set_ylabel('Ventas totales ($)')
ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
for i, (nombre, val) in enumerate(ventas_vend.items()):
    ax5.text(i, val + 500, f'${val/1000:.1f}K', ha='center', fontsize=9, fontweight='bold')

# 6. Distribución del ticket de venta (histograma)
ax6 = fig.add_subplot(2, 3, 6)
ax6.hist(df['total'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax6.axvline(np.mean(df['total']), color='tomato', lw=2, linestyle='--',
             label=f'Media ${np.mean(df["total"]):,.0f}')
ax6.axvline(np.median(df['total']), color='green', lw=2, linestyle=':',
             label=f'Mediana ${np.median(df["total"]):,.0f}')
ax6.set_title('Distribución del ticket de venta', fontweight='bold')
ax6.set_xlabel('Total de venta ($)')
ax6.set_ylabel('Frecuencia')
ax6.legend()

plt.suptitle('📊 Dashboard de Ventas — Análisis Completo con NumPy & pandas',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'\n💡 Insight clave: La mediana (${np.median(df["total"]):,.0f}) es',
      f'menor a la media (${np.mean(df["total"]):,.0f}),',
      'lo que indica que pocas ventas grandes sesgan el promedio hacia arriba.')

In [ ]:
# ✅ SOLUCIÓN — Parte C: Operaciones avanzadas con NumPy

print('=== NumPy avanzado: operaciones sobre arrays 2D ===')

# Creamos una matriz de ventas mensuales por producto (NumPy puro)
pivot_np = df.pivot_table(
    values='total', index='mes', columns='producto', aggfunc='sum', fill_value=0
).values  # .values convierte a ndarray NumPy

print(f'\nMatriz de ventas (meses x productos): shape = {pivot_np.shape}')
print(f'Suma total por mes (axis=1): {np.sum(pivot_np, axis=1).round(0)}')
print(f'Suma total por producto (axis=0): {np.sum(pivot_np, axis=0).round(0)}')

# Normalización: ¿qué % representa cada producto en cada mes?
total_por_mes = np.sum(pivot_np, axis=1, keepdims=True)  # shape (12, 1)
participacion = (pivot_np / total_por_mes * 100).round(1)
print(f'\nParticipación promedio de cada producto: {np.mean(participacion, axis=0).round(1)}')

# Correlación entre productos (¿se venden juntos?)
print('\n=== Correlación entre ventas de productos (mes a mes) ===')
productos_cols = df.pivot_table(
    values='total', index='mes', columns='producto', aggfunc='sum', fill_value=0
).columns.tolist()
corr_matrix = np.corrcoef(pivot_np.T)  # .T transpone para correlacionar productos
df_corr = pd.DataFrame(corr_matrix, index=productos_cols, columns=productos_cols)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix), k=1)  # Ocultar triángulo superior
sns.heatmap(df_corr, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax,
            mask=mask.astype(bool), vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'label': 'Correlación'})
ax.set_title('Correlación entre ventas mensuales de productos\n(verde=positiva, rojo=negativa)',
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📦 Ejercicio 2 — Optimización matemática con SciPy

**Enunciado:** Usá SciPy para resolver problemas de optimización: encontrar valores que maximizan una función sujeta a restricciones.

### 📖 Explicación conceptual

**¿Qué es optimización?**

Optimizar es encontrar los valores de las variables que **minimizan o maximizan** una función objetivo, respetando ciertas **restricciones**.

```
Minimizar:   f(x, y) = costo total
Sujeto a:    x + y ≤ 100  (restricción de capacidad)
             x ≥ 0, y ≥ 0 (no negatividad)
```

**SciPy tiene tres módulos principales para esto:**

| Módulo | Función | Cuándo usar |
|--------|---------|-------------|
| `scipy.optimize.minimize` | Minimización general | Funciones continuas con/sin restricciones |
| `scipy.optimize.linprog` | Programación lineal | Función y restricciones lineales |
| `scipy.optimize.minimize_scalar` | 1 variable | Funciones de una sola variable |

**Tipos de restricciones en `minimize`:**
```python
# Restricción de igualdad: g(x) = 0
{'type': 'eq', 'fun': lambda x: x[0] + x[1] - 100}

# Restricción de desigualdad: g(x) >= 0
{'type': 'ineq', 'fun': lambda x: x[0] - 10}  # x[0] >= 10
```

**Límites (bounds):** definen el rango permitido para cada variable:
```python
bounds = [(0, None), (0, 50)]  # x1 >= 0, 0 <= x2 <= 50
```

### 🏭 Caso 1: Optimización de producción

Una fábrica produce dos productos: A y B.
- **Ganancia:** Producto A = $25/unidad, Producto B = $30/unidad
- **Restricciones:**
  - Horas de máquina: 2A + 3B ≤ 240 horas
  - Horas de trabajo: A + 2B ≤ 160 horas  
  - Mínimo a producir: A ≥ 10, B ≥ 10
- **Pregunta:** ¿Cuántas unidades de A y B maximizan la ganancia?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — Caso 1: Optimización de producción
# Tip: scipy.optimize.minimize MINIMIZA, así que para MAXIMIZAR
# la ganancia 25A + 30B, minimizás el negativo: -(25A + 30B)

from scipy.optimize import minimize

# 1. Definí la función objetivo (negada para maximizar)
# 2. Definí las restricciones como lista de dicts
# 3. Definí los bounds
# 4. Llamá a minimize() y mostrá el resultado


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```python
def ganancia_negativa(x):
    A, B = x
    return -(25*A + 30*B)  # negativo porque minimize minimiza

restricciones = [
    {'type': 'ineq', 'fun': lambda x: 240 - (2*x[0] + 3*x[1])},  # horas máquina
    {'type': 'ineq', 'fun': lambda x: 160 - (x[0] + 2*x[1])},    # horas trabajo
]

bounds = [(10, None), (10, None)]  # A >= 10, B >= 10

resultado = minimize(ganancia_negativa, x0=[20, 20],  # punto inicial
                     bounds=bounds, constraints=restricciones)
```
`resultado.x` contiene los valores óptimos, `resultado.fun` el valor de la función (negado).

</details>

In [ ]:
# ✅ SOLUCIÓN — Caso 1: Optimización de producción
from scipy.optimize import minimize, linprog

print('=' * 55)
print('CASO 1: Maximización de ganancia de producción')
print('=' * 55)

# Función objetivo: MAXIMIZAR 25A + 30B
# minimize() minimiza, así que pasamos el negativo
def objetivo(x):
    A, B = x
    return -(25 * A + 30 * B)  # negativo = maximizar

# Restricciones: todas en la forma g(x) >= 0
restricciones = [
    {'type': 'ineq', 'fun': lambda x: 240 - (2*x[0] + 3*x[1])},  # horas máquina
    {'type': 'ineq', 'fun': lambda x: 160 - (x[0]   + 2*x[1])},  # horas trabajo
]

# Bounds: A ∈ [10, ∞), B ∈ [10, ∞)
bounds = [(10, None), (10, None)]

# Punto inicial (importante para encontrar el óptimo global)
x0 = [20, 20]

resultado = minimize(
    objetivo,
    x0=x0,
    method='SLSQP',       # Método recomendado para problemas con restricciones
    bounds=bounds,
    constraints=restricciones
)

A_opt, B_opt = resultado.x
ganancia_opt = -resultado.fun  # El negativo del mínimo = el máximo

print(f'\n✅ Solución encontrada: {resultado.success}')
print(f'   Mensaje: {resultado.message}')
print(f'\n📦 Producción óptima:')
print(f'   Producto A: {A_opt:.1f} unidades')
print(f'   Producto B: {B_opt:.1f} unidades')
print(f'\n💰 Ganancia máxima: ${ganancia_opt:,.2f}')

# Verificamos que se cumplen las restricciones
print(f'\n✔️  Verificación de restricciones:')
print(f'   Horas máquina usadas: {2*A_opt + 3*B_opt:.1f} / 240 (límite)')
print(f'   Horas trabajo usadas: {A_opt + 2*B_opt:.1f} / 160 (límite)')
print(f'   A >= 10: {A_opt:.1f} ✓')
print(f'   B >= 10: {B_opt:.1f} ✓')

In [ ]:
# ✅ SOLUCIÓN — Visualización de la región factible

fig, ax = plt.subplots(figsize=(9, 7))

# Región factible
A_vals = np.linspace(0, 130, 400)

# Curvas de restricción
B_maq   = (240 - 2*A_vals) / 3      # 2A + 3B = 240  → B = (240-2A)/3
B_trab  = (160 - A_vals)   / 2      # A + 2B = 160   → B = (160-A)/2

# Región factible: debajo de ambas restricciones y sobre los mínimos
B_max_factible = np.minimum(np.maximum(B_maq, 0), np.maximum(B_trab, 0))
B_factible     = np.minimum(B_maq, B_trab)

ax.plot(A_vals, B_maq,  'b-',  lw=2, label='2A + 3B = 240 (máquina)')
ax.plot(A_vals, B_trab, 'g-',  lw=2, label='A + 2B = 160 (trabajo)')
ax.axvline(10, color='orange', lw=1.5, linestyle='--', label='A = 10 (mínimo)')
ax.axhline(10, color='purple', lw=1.5, linestyle='--', label='B = 10 (mínimo)')

# Sombreamos la región factible aproximada
B_fill = np.where(
    (B_factible >= 10) & (A_vals >= 10),
    B_factible, np.nan
)
ax.fill_between(A_vals, 10, B_fill, alpha=0.15, color='blue', label='Región factible')

# Punto óptimo
ax.scatter(A_opt, B_opt, color='red', zorder=5, s=200, marker='*',
           label=f'Óptimo: A={A_opt:.0f}, B={B_opt:.0f}')
ax.annotate(
    f'  Óptimo\n  A={A_opt:.0f}, B={B_opt:.0f}\n  Ganancia=${ganancia_opt:,.0f}',
    (A_opt, B_opt), fontsize=10, color='red', fontweight='bold'
)

# Curvas de nivel de la función objetivo (isoganancias)
for g in [1000, 1500, 2000, 2500, 3000]:
    B_iso = (g - 25*A_vals) / 30
    mask  = (B_iso >= 0) & (A_vals >= 0)
    ax.plot(A_vals[mask], B_iso[mask], 'r:', alpha=0.3, lw=1)
    idx = np.argmax(B_iso[mask])

ax.set_xlim(0, 130)
ax.set_ylim(0, 90)
ax.set_xlabel('Unidades de Producto A', fontsize=12)
ax.set_ylabel('Unidades de Producto B', fontsize=12)
ax.set_title('Región factible y punto óptimo de producción\n(líneas punteadas rojas = curvas de ganancia constante)',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 📈 Caso 2: Ajuste de curva y optimización de precios

Tenemos datos de demanda a distintos precios. Queremos:
1. Ajustar una curva de demanda a los datos.
2. Encontrar el precio que **maximiza el ingreso** (precio × cantidad).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ — Caso 2: Optimización de precio
# Datos: precio y demanda observada
precios  = np.array([10, 15, 20, 25, 30, 35, 40, 50, 60, 80])
demanda  = np.array([500, 420, 350, 290, 240, 195, 155, 100, 65, 30])

# 1. Usá scipy.optimize.curve_fit para ajustar una curva de demanda
#    Probá un modelo exponencial: demanda = a * exp(-b * precio)
#    o un modelo potencia:        demanda = a * precio^(-b)
# 2. Calculá el ingreso = precio * demanda_predicha para cada precio
# 3. Usá minimize_scalar para encontrar el precio óptimo
# 4. Graficá los datos, la curva ajustada y el punto óptimo


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```python
from scipy.optimize import curve_fit, minimize_scalar

# Modelo exponencial de demanda
def modelo_demanda(precio, a, b):
    return a * np.exp(-b * precio)

# Ajustar el modelo a los datos
params, _ = curve_fit(modelo_demanda, precios, demanda, p0=[800, 0.05])
a_opt, b_opt = params

# Función de ingreso negado (para minimizar = maximizar ingreso)
def ingreso_negado(p):
    return -(p * modelo_demanda(p, a_opt, b_opt))

# Encontrar el precio óptimo
resultado = minimize_scalar(ingreso_negado, bounds=(10, 100), method='bounded')
precio_optimo = resultado.x
```

</details>

In [ ]:
# ✅ SOLUCIÓN — Caso 2: Optimización de precios
from scipy.optimize import curve_fit, minimize_scalar

precios = np.array([10, 15, 20, 25, 30, 35, 40, 50, 60, 80])
demanda = np.array([500, 420, 350, 290, 240, 195, 155, 100, 65, 30])

print('=' * 55)
print('CASO 2: Optimización de precio para máximo ingreso')
print('=' * 55)

# Modelo de demanda: exponencial D(p) = a * e^(-b*p)
def modelo_demanda(p, a, b):
    return a * np.exp(-b * p)

# curve_fit ajusta los parámetros a, b a los datos reales
params, covarianza = curve_fit(modelo_demanda, precios, demanda, p0=[800, 0.04])
a_fit, b_fit = params
error_params  = np.sqrt(np.diag(covarianza))  # Incertidumbre de cada parámetro

print(f'\n📐 Modelo ajustado: D(p) = {a_fit:.1f} × e^(-{b_fit:.4f} × p)')
print(f'   Error en a: ±{error_params[0]:.1f} | Error en b: ±{error_params[1]:.5f}')

# Bondad del ajuste (R²)
demanda_pred = modelo_demanda(precios, a_fit, b_fit)
ss_res = np.sum((demanda - demanda_pred)**2)
ss_tot = np.sum((demanda - np.mean(demanda))**2)
r2_ajuste = 1 - ss_res/ss_tot
print(f'   R² del ajuste: {r2_ajuste:.4f}')

# Función de ingreso: I(p) = p × D(p)
def ingreso(p):
    return p * modelo_demanda(p, a_fit, b_fit)

def ingreso_negado(p):
    return -ingreso(p)

# Optimización: encontrar precio que maximiza el ingreso
resultado = minimize_scalar(
    ingreso_negado,
    bounds=(5, 120),
    method='bounded'
)

p_opt = resultado.x
d_opt = modelo_demanda(p_opt, a_fit, b_fit)
i_opt = ingreso(p_opt)

print(f'\n🏆 Precio óptimo: ${p_opt:.2f}')
print(f'   Demanda esperada: {d_opt:.0f} unidades')
print(f'   Ingreso máximo:  ${i_opt:,.2f}')

# Comparación con precios actuales
print(f'\n📊 Comparación de ingresos a distintos precios:')
for p_prueba in [15, 20, p_opt, 40, 60]:
    ing = ingreso(p_prueba)
    es_opt = " ← ÓPTIMO" if abs(p_prueba - p_opt) < 1 else ""
    print(f'   Precio ${p_prueba:>5.1f}: demanda={modelo_demanda(p_prueba,a_fit,b_fit):.0f}, ingreso=${ing:>8,.0f}{es_opt}')

# Visualización
p_range = np.linspace(5, 100, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Curva de demanda
axes[0].scatter(precios, demanda, color='steelblue', s=80, zorder=5, label='Datos observados')
axes[0].plot(p_range, modelo_demanda(p_range, a_fit, b_fit),
             color='tomato', lw=2, label=f'Ajuste exponencial (R²={r2_ajuste:.3f})')
axes[0].scatter(p_opt, d_opt, color='red', s=150, marker='*', zorder=6, label=f'Precio óptimo ${p_opt:.1f}')
axes[0].axvline(p_opt, color='red', linestyle='--', alpha=0.5)
axes[0].set_title(f'Curva de demanda ajustada\nD(p) = {a_fit:.0f} × e^(-{b_fit:.4f}p)', fontweight='bold')
axes[0].set_xlabel('Precio ($)')
axes[0].set_ylabel('Demanda (unidades)')
axes[0].legend()

# Gráfico 2: Curva de ingreso
ingresos_range = ingreso(p_range)
axes[1].plot(p_range, ingresos_range, color='seagreen', lw=2.5, label='Ingreso total')
axes[1].scatter(p_opt, i_opt, color='red', s=200, marker='*', zorder=5,
               label=f'Máximo: ${i_opt:,.0f} a p=${p_opt:.1f}')
axes[1].axvline(p_opt, color='red', linestyle='--', alpha=0.6)
axes[1].axhline(i_opt, color='red', linestyle=':', alpha=0.4)

# Sombreamos el área bajo la curva en la zona óptima
mask_opt = (p_range >= p_opt - 5) & (p_range <= p_opt + 5)
axes[1].fill_between(p_range[mask_opt], ingresos_range[mask_opt], alpha=0.2, color='red')

axes[1].set_title(f'Curva de ingreso total\nI(p) = p × D(p)', fontweight='bold')
axes[1].set_xlabel('Precio ($)')
axes[1].set_ylabel('Ingreso total ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].legend()

plt.suptitle('📈 Optimización de precio para máximo ingreso', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 🧮 Caso 3: Optimización de portfolio de inversión (bonus)

Tenemos un presupuesto de $10,000 para distribuir entre 3 activos. Cada uno tiene un retorno esperado y un riesgo. Queremos **maximizar el retorno minimizando el riesgo**.

In [ ]:
# ✅ SOLUCIÓN — Caso 3: Optimización de portfolio

print('=' * 55)
print('CASO 3: Optimización de portfolio de inversión')
print('=' * 55)

# Datos del portfolio
activos    = ['Acciones', 'Bonos', 'Inmuebles']
retornos   = np.array([0.12, 0.06, 0.09])   # Retorno anual esperado
riesgos    = np.array([0.20, 0.05, 0.12])   # Desviación estándar (riesgo)
presupuesto = 10_000

# Función objetivo: minimizar riesgo dado un retorno mínimo requerido
# Variables: w = pesos (% del portfolio en cada activo)

def riesgo_portfolio(w):
    """Riesgo total = suma ponderada de riesgos individuales (simplificado)."""
    return np.sum(w * riesgos)

# Restricciones
restricciones_port = [
    # Los pesos deben sumar 1 (100% del presupuesto invertido)
    {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
    # Retorno mínimo del 8%
    {'type': 'ineq', 'fun': lambda w: np.dot(w, retornos) - 0.08},
]

# Bounds: cada activo entre 0% y 80% del portfolio
bounds_port = [(0.0, 0.8)] * 3

# Punto inicial: distribución equitativa
w0 = np.array([1/3, 1/3, 1/3])

resultado_port = minimize(
    riesgo_portfolio,
    x0=w0,
    method='SLSQP',
    bounds=bounds_port,
    constraints=restricciones_port
)

w_opt = resultado_port.x
retorno_opt = np.dot(w_opt, retornos)
riesgo_opt  = riesgo_portfolio(w_opt)

print(f'\n✅ Portfolio óptimo (mínimo riesgo con retorno ≥ 8%):')
print(f"{'Activo':<12} {'Peso %':>8} {'Inversión $':>12} {'Retorno esp.':>14}")
print('-' * 50)
for activo, w, ret in zip(activos, w_opt, retornos):
    print(f'{activo:<12} {w*100:>7.1f}% {w*presupuesto:>11,.0f}  {w*ret*100:>11.2f}%')
print('-' * 50)
print(f'{"TOTAL":<12} {100:>7.0f}% {presupuesto:>11,.0f}  {retorno_opt*100:>11.2f}%')
print(f'\n   Retorno esperado: {retorno_opt*100:.2f}%')
print(f'   Riesgo total:     {riesgo_opt*100:.2f}%')
print(f'   Ratio retorno/riesgo: {retorno_opt/riesgo_opt:.2f}')

# Visualización del portfolio
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Torta del portfolio óptimo
colores_port = ['#4CAF50', '#2196F3', '#FF9800']
wedges, texts, autos = axes[0].pie(
    w_opt * 100, labels=activos, autopct='%1.1f%%',
    colors=colores_port, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title(f'Portfolio óptimo\nRetorno={retorno_opt*100:.1f}% | Riesgo={riesgo_opt*100:.1f}%',
                  fontweight='bold')

# Frontera eficiente (distintos niveles de retorno mínimo requerido)
retornos_min   = np.linspace(0.062, 0.115, 30)
riesgos_front  = []
retornos_front = []

for ret_min in retornos_min:
    rest = [
        {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
        {'type': 'ineq', 'fun': lambda w, r=ret_min: np.dot(w, retornos) - r},
    ]
    res = minimize(riesgo_portfolio, w0, method='SLSQP',
                   bounds=bounds_port, constraints=rest)
    if res.success:
        riesgos_front.append(res.fun * 100)
        retornos_front.append(np.dot(res.x, retornos) * 100)

axes[1].plot(riesgos_front, retornos_front, 'b-o', markersize=4, lw=2, label='Frontera eficiente')
axes[1].scatter(riesgo_opt*100, retorno_opt*100, color='red', s=200, marker='*',
               zorder=5, label='Portfolio seleccionado')

# Activos individuales
for activo, r, ret, c in zip(activos, riesgos*100, retornos*100, colores_port):
    axes[1].scatter(r, ret, color=c, s=100, zorder=5)
    axes[1].annotate(activo, (r, ret), textcoords='offset points',
                     xytext=(5, 5), fontsize=9)

axes[1].set_title('Frontera eficiente del portfolio\n(máximo retorno para cada nivel de riesgo)',
                  fontweight='bold')
axes[1].set_xlabel('Riesgo (%)')
axes[1].set_ylabel('Retorno esperado (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('💼 Optimización de Portfolio de Inversión', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🏆 Resumen del notebook

| Ejercicio | Tema | Herramientas |
|-----------|------|--------------|
| 1A | Análisis de ventas con pandas | `groupby`, `pivot_table`, `describe` |
| 1B | Dashboard de 6 visualizaciones | `matplotlib`, `seaborn`, `mticker` |
| 1C | Operaciones matriciales con NumPy | Arrays 2D, `corrcoef`, `axis` |
| 2A | Optimización de producción | `minimize`, SLSQP, región factible |
| 2B | Ajuste de curva + precio óptimo | `curve_fit`, `minimize_scalar` |
| 2C | Optimización de portfolio | Frontera eficiente, restricciones múltiples |

### 🔑 Diferencia clave entre NumPy y pandas

| | NumPy | pandas |
|--|-------|--------|
| **Estructura** | ndarray (sin etiquetas) | DataFrame/Series (con etiquetas) |
| **Velocidad** | Más rápido (operaciones de bajo nivel) | Más conveniente para datos tabulares |
| **Uso típico** | Álgebra lineal, cálculo numérico | Análisis de datos, ETL, estadística |
| **Relación** | pandas está construido sobre NumPy | `df.values` convierte a ndarray |

### 💪 Desafíos extra
1. **Ej. 1:** Analizá las ventas por región. ¿Hay estacionalidad diferente en cada región?
2. **Ej. 1:** Usá `np.polyfit()` para ajustar una tendencia lineal a las ventas mensuales.
3. **Ej. 2A:** Agregá un tercer producto C con ganancia $20 y 1.5 horas de máquina + 1 hora de trabajo.
4. **Ej. 2B:** Probá también un modelo de demanda lineal `D(p) = a - b*p` y comparalos por R².
5. **Ej. 2C:** Agregá un cuarto activo (Criptomonedas, retorno=20%, riesgo=50%) y observá cómo cambia la frontera.